In [1]:
# -*- coding: utf-8 -*-
"""
Created on Sun Mar 22 22:04:05 2026

@author: Sourav
"""

# =========================================
# RHC-UCRL (Correct Version with 4 Critics)
# =========================================

import torch
import torch.nn as nn
import numpy as np
import gymnasium as gym
from collections import deque
import random
import copy
from tqdm import tqdm
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================
# Replay Buffer
# =========================================
class Buffer:
    def __init__(self, size=100000):
        self.buffer = deque(maxlen=size)

    def add(self, s, a, adv, r, c, s_next):
        self.buffer.append((s, a, adv, r, c, s_next))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, adv, r, c, s_next = zip(*batch)

        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(np.array(a), dtype=torch.float32, device=device),
            torch.tensor(np.array(adv), dtype=torch.float32, device=device),
            torch.tensor(np.array(r), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(c), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(s_next), dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# =========================================
# Networks
# =========================================
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, a_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return 2 * self.net(s)


class Adversary(nn.Module):
    def __init__(self, s_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, adv_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return self.net(s)


class Critic(nn.Module):
    def __init__(self, s_dim, a_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim + a_dim + adv_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, s, a, adv):
        return self.net(torch.cat([s, a, adv], dim=-1))


# =========================================
# Ensemble for hallucination
# =========================================
class DynamicsModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class Ensemble:
    def __init__(self, in_dim, hidden, out_dim, K=5):
        self.models = [DynamicsModel(in_dim, hidden, out_dim).to(device) for _ in range(K)]
        self.opts = [torch.optim.Adam(m.parameters(), lr=1e-3) for m in self.models]

    def predict(self, s, a, adv):
        x = torch.cat([s, a, adv], dim=-1)
        preds = torch.stack([m(x) for m in self.models])
        mu = preds.mean(0)
        sigma = preds.std(0)
        return mu, sigma

    def train(self, S, A, ADV, S_next):
        delta = S_next - S
        losses = []

        for model, opt in zip(self.models, self.opts):

            # Trick 6: bootstrapping samples 
            idx = torch.randint(0, S.shape[0], (S.shape[0]//2,), device=device)

            s_i = S[idx]
            a_i = A[idx]
            adv_i = ADV[idx]
            d_i = delta[idx] + 0.01 * torch.randn_like(delta[idx])

            x = torch.cat([s_i, a_i, adv_i], dim=-1)

            pred = model(x)
            loss = ((pred - d_i) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            losses.append(loss.item())

        return losses


# =========================================
# Adversarial Pendulum
# =========================================
class AdversarialPendulum:
    def __init__(self):
        self.env = gym.make("Pendulum-v1")

    def reset(self):
        s, _ = self.env.reset()
        return s

    def step(self, s, a, adv):
        s_next, _, done, trunc, _ = self.env.step(a)

        fx, fy = adv
        cos_t, sin_t, theta_dot = s_next
        theta = np.arctan2(sin_t, cos_t)

        theta += 0.05 * fx
        theta_dot += 0.05 * fy

        s_next = np.array([np.cos(theta), np.sin(theta), theta_dot], dtype=np.float32)

        reward = -(theta**2 + 0.1 * theta_dot**2 + 0.001 * a.squeeze()**2)/50  #Trick 3: Reward Normalization

        # constraint: keep near upright
        cost = (abs(theta) > 0.5).astype(np.float32)

        return s_next, reward, cost, done or trunc


# =========================================
# Setup
# =========================================
env = AdversarialPendulum()
buffer = Buffer()

s_dim, a_dim, adv_dim = 3, 1, 2

actor = Actor(s_dim, a_dim).to(device)
adversary = Adversary(s_dim, adv_dim).to(device)

# 4 critics
critic_r_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_o_target = copy.deepcopy(critic_r_o)
critic_r_p_target = copy.deepcopy(critic_r_p)
critic_u_o_target = copy.deepcopy(critic_u_o)
critic_u_p_target = copy.deepcopy(critic_u_p)

opt = lambda net: torch.optim.Adam(net.parameters(), lr=1e-3)

opt_actor = opt(actor)
opt_adv = opt(adversary)
opt_r_o = opt(critic_r_o)
opt_r_p = opt(critic_r_p)
opt_u_o = opt(critic_u_o)
opt_u_p = opt(critic_u_p)

ensemble = Ensemble(s_dim + a_dim + adv_dim, 64, s_dim)

gamma = 0.99
beta, alpha = 0.1, 0.1
b = 60
lambda_ = torch.tensor(50.0, device=device)
tau = 0.005  # soft update
reward= []
cost = []

def soft_update(net, target_net):
    for p, tp in zip(net.parameters(), target_net.parameters()):
        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)


# =========================================
# Training Loop
# =========================================
for ep in tqdm(range(1000)):

    s = env.reset()
    total_r = 0
    total_c = 0

    for t in range(200):
        s_t = torch.tensor(s, dtype=torch.float32, device=device)

        with torch.no_grad():
            a = actor(s_t).cpu().numpy()
            # Trick 7: Add exploration noise
            noise = np.random.normal(0, 0.2, size=a.shape)
            a = a + noise
            a = np.clip(a, -2, 2)
            adv = adversary(s_t).cpu().numpy()
            # Trick 8: mild adversarial noise
            adv_noise = np.random.normal(0, 0.1, size=adv.shape)
            adv = adv + adv_noise
            adv = np.clip(adv, -1, 1)

        s_next, r, c, done = env.step(s, a, adv)

        buffer.add(s, a, adv, r, c, s_next)
        s = s_next
        total_r += r
        total_c += c

        if len(buffer) > 5000: #Fix buffer length
            S, A, ADV, R, C, S_next = buffer.sample(400) #Fix the samples used
            ensemble.train(S, A, ADV, S_next)

            with torch.no_grad():
                A_next = actor(S_next)
                ADV_next = adversary(S_next)

                _, sigma = ensemble.predict(S_next, A_next, ADV_next)
                sig = sigma.mean(dim=1, keepdim=True)

                # optimistic targets
                #critic_r_o_target = deepcopy(critic_r_o) ##Trick 1 separate updation
                #critic_r_p_target = deepcopy(critic_r_p)  ##Trick 1
                target_r_o = R + beta * sig + gamma * critic_r_o_target(S_next, A_next, ADV_next)
                target_u_o = C + alpha * sig + gamma * critic_u_o_target(S_next, A_next, ADV_next)

                # pessimistic targets
                target_r_p = R - beta * sig + gamma * critic_r_p_target(S_next, A_next, ADV_next)
                target_u_p = C - alpha * sig + gamma * critic_u_p_target(S_next, A_next, ADV_next)

            # update critics
            loss = lambda net, tgt: ((net(S, A, ADV) - tgt)**2).mean()

            opt_r_o.zero_grad(); loss(critic_r_o, target_r_o).backward(); opt_r_o.step()
            opt_r_p.zero_grad(); loss(critic_r_p, target_r_p).backward(); opt_r_p.step()
            opt_u_o.zero_grad(); loss(critic_u_o, target_u_o).backward(); opt_u_o.step()
            opt_u_p.zero_grad(); loss(critic_u_p, target_u_p).backward(); opt_u_p.step()
            
            ##TRICK 2: Perform sequential soft-updates
            soft_update(critic_r_o, critic_r_o_target)
            soft_update(critic_r_p, critic_r_p_target)
            soft_update(critic_u_o, critic_u_o_target)
            soft_update(critic_u_p, critic_u_p_target)
            
            torch.nn.utils.clip_grad_norm_(critic_r_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_r_p.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_p.parameters(), 1.0)
            

            # ===== Actor (optimistic) =====
            A_pred = actor(S)
            ADV_fixed = adversary(S).detach()

            q_r = critic_r_o(S, A_pred, ADV_fixed)
            q_u = critic_u_o(S, A_pred, ADV_fixed)

            actor_loss = -(q_r - lambda_ * torch.relu(q_u - b)).mean()

            opt_actor.zero_grad()
            actor_loss.backward()
            opt_actor.step()

            # ===== Adversary (pessimistic) =====
            ADV_pred = adversary(S)
            A_fixed = actor(S).detach()

            q_r = critic_r_p(S, A_fixed, ADV_pred)
            q_u = critic_u_p(S, A_fixed, ADV_pred)
            
            if t%10==0:  ## Trick 4: Use adversarial training mildly
                adv_loss = (q_r - lambda_ * torch.relu(q_u - b)).mean()
                opt_adv.zero_grad()
                adv_loss.backward()
                opt_adv.step()

            # # ===== Lambda update =====
            # with torch.no_grad():
            #     violation = critic_u_p(S, A, ADV) - b
            #     lambda_ += 0.01 * violation.mean()
            #     lambda_ = torch.clamp(lambda_, 0, 10)

        if done:
            break

    #print(f"Ep {ep}, Reward: {total_r*20:.2f}, Cost/Utility: {total_c:.2f}, Lambda: {lambda_.item():.3f}")
    reward.append(total_r*20)
    cost.append(total_c)
df = {'Reward':reward,'Cost/utility':cost}
Df = pd.DataFrame(df)
Df.to_excel('Pendulumv1_strict_violation_smaller_reward.xlsx')

100%|██████████| 1000/1000 [42:01<00:00,  2.52s/it]


In [2]:
# -*- coding: utf-8 -*-
"""
Created on Sun Mar 22 22:04:05 2026

@author: Sourav
"""

# =========================================
# RHC-UCRL (Correct Version with 4 Critics)
# =========================================

import torch
import torch.nn as nn
import numpy as np
import gymnasium as gym
from collections import deque
import random
import copy
from tqdm import tqdm
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================
# Replay Buffer
# =========================================
class Buffer:
    def __init__(self, size=100000):
        self.buffer = deque(maxlen=size)

    def add(self, s, a, adv, r, c, s_next):
        self.buffer.append((s, a, adv, r, c, s_next))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, adv, r, c, s_next = zip(*batch)

        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(np.array(a), dtype=torch.float32, device=device),
            torch.tensor(np.array(adv), dtype=torch.float32, device=device),
            torch.tensor(np.array(r), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(c), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(s_next), dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# =========================================
# Networks
# =========================================
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, a_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return 2 * self.net(s)


class Adversary(nn.Module):
    def __init__(self, s_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, adv_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return self.net(s)


class Critic(nn.Module):
    def __init__(self, s_dim, a_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim + a_dim + adv_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, s, a, adv):
        return self.net(torch.cat([s, a, adv], dim=-1))


# =========================================
# Ensemble for hallucination
# =========================================
class DynamicsModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class Ensemble:
    def __init__(self, in_dim, hidden, out_dim, K=5):
        self.models = [DynamicsModel(in_dim, hidden, out_dim).to(device) for _ in range(K)]
        self.opts = [torch.optim.Adam(m.parameters(), lr=1e-3) for m in self.models]

    def predict(self, s, a, adv):
        x = torch.cat([s, a, adv], dim=-1)
        preds = torch.stack([m(x) for m in self.models])
        mu = preds.mean(0)
        sigma = preds.std(0)
        return mu, sigma

    def train(self, S, A, ADV, S_next):
        delta = S_next - S
        losses = []

        for model, opt in zip(self.models, self.opts):

            # Trick 6: bootstrapping samples 
            idx = torch.randint(0, S.shape[0], (S.shape[0]//2,), device=device)

            s_i = S[idx]
            a_i = A[idx]
            adv_i = ADV[idx]
            d_i = delta[idx] + 0.01 * torch.randn_like(delta[idx])

            x = torch.cat([s_i, a_i, adv_i], dim=-1)

            pred = model(x)
            loss = ((pred - d_i) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            losses.append(loss.item())

        return losses


# =========================================
# Adversarial Pendulum
# =========================================
class AdversarialPendulum:
    def __init__(self):
        self.env = gym.make("Pendulum-v1")

    def reset(self):
        s, _ = self.env.reset()
        return s

    def step(self, s, a, adv):
        s_next, _, done, trunc, _ = self.env.step(a)

        fx, fy = adv
        cos_t, sin_t, theta_dot = s_next
        theta = np.arctan2(sin_t, cos_t)

        theta += 0.05 * fx
        theta_dot += 0.05 * fy

        s_next = np.array([np.cos(theta), np.sin(theta), theta_dot], dtype=np.float32)

        reward = -(theta**2 + 0.1 * theta_dot**2 + 0.001 * a.squeeze()**2)/50  #Trick 3: Reward Normalization

        # constraint: keep near upright
        cost = (abs(theta) > 0.5).astype(np.float32)

        return s_next, reward, cost, done or trunc


# =========================================
# Setup
# =========================================
env = AdversarialPendulum()
buffer = Buffer()

s_dim, a_dim, adv_dim = 3, 1, 2

actor = Actor(s_dim, a_dim).to(device)
adversary = Adversary(s_dim, adv_dim).to(device)

# 4 critics
critic_r_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_o_target = copy.deepcopy(critic_r_o)
critic_r_p_target = copy.deepcopy(critic_r_p)
critic_u_o_target = copy.deepcopy(critic_u_o)
critic_u_p_target = copy.deepcopy(critic_u_p)

opt = lambda net: torch.optim.Adam(net.parameters(), lr=1e-3)

opt_actor = opt(actor)
opt_adv = opt(adversary)
opt_r_o = opt(critic_r_o)
opt_r_p = opt(critic_r_p)
opt_u_o = opt(critic_u_o)
opt_u_p = opt(critic_u_p)

ensemble = Ensemble(s_dim + a_dim + adv_dim, 64, s_dim)

gamma = 0.99
beta, alpha = 0.1, 0.1
b = 60
lambda_ = torch.tensor(50.0, device=device)
tau = 0.005  # soft update
reward= []
cost = []

def soft_update(net, target_net):
    for p, tp in zip(net.parameters(), target_net.parameters()):
        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)


# =========================================
# Training Loop
# =========================================
for ep in tqdm(range(1000)):

    s = env.reset()
    total_r = 0
    total_c = 0

    for t in range(200):
        s_t = torch.tensor(s, dtype=torch.float32, device=device)

        with torch.no_grad():
            a = actor(s_t).cpu().numpy()
            # Trick 7: Add exploration noise
            noise = np.random.normal(0, 0.2, size=a.shape)
            a = a + noise
            a = np.clip(a, -2, 2)
            adv = adversary(s_t).cpu().numpy()
            # Trick 8: mild adversarial noise
            adv_noise = np.random.normal(0, 0.01, size=adv.shape)
            adv = adv + adv_noise
            adv = np.clip(adv, -1, 1)

        s_next, r, c, done = env.step(s, a, adv)

        buffer.add(s, a, adv, r, c, s_next)
        s = s_next
        total_r += r
        total_c += c

        if len(buffer) > 5000: #Fix buffer length
            S, A, ADV, R, C, S_next = buffer.sample(400) #Fix the samples used
            ensemble.train(S, A, ADV, S_next)

            with torch.no_grad():
                A_next = actor(S_next)
                ADV_next = adversary(S_next)

                _, sigma = ensemble.predict(S_next, A_next, ADV_next)
                sig = sigma.mean(dim=1, keepdim=True)

                # optimistic targets
                #critic_r_o_target = deepcopy(critic_r_o) ##Trick 1 separate updation
                #critic_r_p_target = deepcopy(critic_r_p)  ##Trick 1
                target_r_o = R + beta * sig + gamma * critic_r_o_target(S_next, A_next, ADV_next)
                target_u_o = C + alpha * sig + gamma * critic_u_o_target(S_next, A_next, ADV_next)

                # pessimistic targets
                target_r_p = R - beta * sig + gamma * critic_r_p_target(S_next, A_next, ADV_next)
                target_u_p = C - alpha * sig + gamma * critic_u_p_target(S_next, A_next, ADV_next)

            # update critics
            loss = lambda net, tgt: ((net(S, A, ADV) - tgt)**2).mean()

            opt_r_o.zero_grad(); loss(critic_r_o, target_r_o).backward(); opt_r_o.step()
            opt_r_p.zero_grad(); loss(critic_r_p, target_r_p).backward(); opt_r_p.step()
            opt_u_o.zero_grad(); loss(critic_u_o, target_u_o).backward(); opt_u_o.step()
            opt_u_p.zero_grad(); loss(critic_u_p, target_u_p).backward(); opt_u_p.step()
            
            ##TRICK 2: Perform sequential soft-updates
            soft_update(critic_r_o, critic_r_o_target)
            soft_update(critic_r_p, critic_r_p_target)
            soft_update(critic_u_o, critic_u_o_target)
            soft_update(critic_u_p, critic_u_p_target)
            
            torch.nn.utils.clip_grad_norm_(critic_r_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_r_p.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_p.parameters(), 1.0)
            

            # ===== Actor (optimistic) =====
            A_pred = actor(S)
            ADV_fixed = adversary(S).detach()

            q_r = critic_r_o(S, A_pred, ADV_fixed)
            q_u = critic_u_o(S, A_pred, ADV_fixed)

            actor_loss = -(q_r - lambda_ * torch.relu(q_u - b)).mean()

            opt_actor.zero_grad()
            actor_loss.backward()
            opt_actor.step()

            # ===== Adversary (pessimistic) =====
            ADV_pred = adversary(S)
            A_fixed = actor(S).detach()

            q_r = critic_r_p(S, A_fixed, ADV_pred)
            q_u = critic_u_p(S, A_fixed, ADV_pred)
            
            if t%20==0:  ## Trick 4: Use adversarial training mildly
                adv_loss = (q_r - lambda_ * torch.relu(q_u - b)).mean()
                opt_adv.zero_grad()
                adv_loss.backward()
                opt_adv.step()

            # # ===== Lambda update =====
            # with torch.no_grad():
            #     violation = critic_u_p(S, A, ADV) - b
            #     lambda_ += 0.01 * violation.mean()
            #     lambda_ = torch.clamp(lambda_, 0, 10)

        if done:
            break

    #print(f"Ep {ep}, Reward: {total_r*20:.2f}, Cost/Utility: {total_c:.2f}, Lambda: {lambda_.item():.3f}")
    reward.append(total_r*20)
    cost.append(total_c)
df = {'Reward':reward,'Cost/utility':cost}
Df = pd.DataFrame(df)
Df.to_excel('Pendulumv1_strict_violation_lower_adversarial_effect.xlsx')

100%|██████████| 1000/1000 [42:00<00:00,  2.52s/it]


In [3]:
# -*- coding: utf-8 -*-
"""
Created on Sun Mar 22 22:04:05 2026

@author: Sourav
"""

# =========================================
# RHC-UCRL (Correct Version with 4 Critics)
# =========================================

import torch
import torch.nn as nn
import numpy as np
import gymnasium as gym
from collections import deque
import random
import copy
from tqdm import tqdm
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================
# Replay Buffer
# =========================================
class Buffer:
    def __init__(self, size=100000):
        self.buffer = deque(maxlen=size)

    def add(self, s, a, adv, r, c, s_next):
        self.buffer.append((s, a, adv, r, c, s_next))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, adv, r, c, s_next = zip(*batch)

        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(np.array(a), dtype=torch.float32, device=device),
            torch.tensor(np.array(adv), dtype=torch.float32, device=device),
            torch.tensor(np.array(r), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(c), dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(s_next), dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


# =========================================
# Networks
# =========================================
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, a_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return 2 * self.net(s)


class Adversary(nn.Module):
    def __init__(self, s_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128),
            nn.ReLU(),
            nn.Linear(128, adv_dim),
            nn.Tanh()
        )

    def forward(self, s):
        return self.net(s)


class Critic(nn.Module):
    def __init__(self, s_dim, a_dim, adv_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim + a_dim + adv_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, s, a, adv):
        return self.net(torch.cat([s, a, adv], dim=-1))


# =========================================
# Ensemble for hallucination
# =========================================
class DynamicsModel(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class Ensemble:
    def __init__(self, in_dim, hidden, out_dim, K=5):
        self.models = [DynamicsModel(in_dim, hidden, out_dim).to(device) for _ in range(K)]
        self.opts = [torch.optim.Adam(m.parameters(), lr=1e-3) for m in self.models]

    def predict(self, s, a, adv):
        x = torch.cat([s, a, adv], dim=-1)
        preds = torch.stack([m(x) for m in self.models])
        mu = preds.mean(0)
        sigma = preds.std(0)
        return mu, sigma

    def train(self, S, A, ADV, S_next):
        delta = S_next - S
        losses = []

        for model, opt in zip(self.models, self.opts):

            # Trick 6: bootstrapping samples 
            idx = torch.randint(0, S.shape[0], (S.shape[0]//2,), device=device)

            s_i = S[idx]
            a_i = A[idx]
            adv_i = ADV[idx]
            d_i = delta[idx] + 0.01 * torch.randn_like(delta[idx])

            x = torch.cat([s_i, a_i, adv_i], dim=-1)

            pred = model(x)
            loss = ((pred - d_i) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            losses.append(loss.item())

        return losses


# =========================================
# Adversarial Pendulum
# =========================================
class AdversarialPendulum:
    def __init__(self):
        self.env = gym.make("Pendulum-v1")

    def reset(self):
        s, _ = self.env.reset()
        return s

    def step(self, s, a, adv):
        s_next, _, done, trunc, _ = self.env.step(a)

        fx, fy = adv
        cos_t, sin_t, theta_dot = s_next
        theta = np.arctan2(sin_t, cos_t)

        theta += 0.05 * fx
        theta_dot += 0.05 * fy

        s_next = np.array([np.cos(theta), np.sin(theta), theta_dot], dtype=np.float32)

        reward = -(theta**2 + 0.1 * theta_dot**2 + 0.001 * a.squeeze()**2)/20  #Trick 3: Reward Normalization

        # constraint: keep near upright
        cost = (abs(theta) > 0.5).astype(np.float32)

        return s_next, reward, cost, done or trunc


# =========================================
# Setup
# =========================================
env = AdversarialPendulum()
buffer = Buffer()

s_dim, a_dim, adv_dim = 3, 1, 2

actor = Actor(s_dim, a_dim).to(device)
adversary = Adversary(s_dim, adv_dim).to(device)

# 4 critics
critic_r_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_o = Critic(s_dim, a_dim, adv_dim).to(device)
critic_u_p = Critic(s_dim, a_dim, adv_dim).to(device)
critic_r_o_target = copy.deepcopy(critic_r_o)
critic_r_p_target = copy.deepcopy(critic_r_p)
critic_u_o_target = copy.deepcopy(critic_u_o)
critic_u_p_target = copy.deepcopy(critic_u_p)

opt = lambda net: torch.optim.Adam(net.parameters(), lr=1e-3)

opt_actor = torch.optim.Adam(actor.parameters(), lr=1e-4)
opt_adv = opt(adversary)
opt_r_o = opt(critic_r_o)
opt_r_p = opt(critic_r_p)
opt_u_o = opt(critic_u_o)
opt_u_p = opt(critic_u_p)

ensemble = Ensemble(s_dim + a_dim + adv_dim, 64, s_dim)

gamma = 0.99
beta, alpha = 0.01, 0.1
b = 60
lambda_ = torch.tensor(50.0, device=device)
tau = 0.001  # soft update
reward= []
cost = []

def soft_update(net, target_net):
    for p, tp in zip(net.parameters(), target_net.parameters()):
        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)


# =========================================
# Training Loop
# =========================================
for ep in tqdm(range(1000)):

    s = env.reset()
    total_r = 0
    total_c = 0

    for t in range(200):
        s_t = torch.tensor(s, dtype=torch.float32, device=device)

        with torch.no_grad():
            a = actor(s_t).cpu().numpy()
            # Trick 7: Add exploration noise
            noise = np.random.normal(0, 0.05, size=a.shape)
            a = a + noise
            a = np.clip(a, -2, 2)
            adv = adversary(s_t).cpu().numpy()
            # Trick 8: mild adversarial noise
            adv_noise = np.random.normal(0, 0.1, size=adv.shape)
            adv = adv + adv_noise
            adv = np.clip(adv, -1, 1)

        s_next, r, c, done = env.step(s, a, adv)

        buffer.add(s, a, adv, r, c, s_next)
        s = s_next
        total_r += r
        total_c += c

        if len(buffer) > 5000: #Fix buffer length
            S, A, ADV, R, C, S_next = buffer.sample(400) #Fix the samples used
            ensemble.train(S, A, ADV, S_next)

            with torch.no_grad():
                A_next = actor(S_next)
                ADV_next = adversary(S_next)

                _, sigma = ensemble.predict(S_next, A_next, ADV_next)
                sig = sigma.mean(dim=1, keepdim=True)

                # optimistic targets
                #critic_r_o_target = deepcopy(critic_r_o) ##Trick 1 separate updation
                #critic_r_p_target = deepcopy(critic_r_p)  ##Trick 1
                target_r_o = R + beta * sig + gamma * critic_r_o_target(S_next, A_next, ADV_next)
                target_u_o = C + alpha * sig + gamma * critic_u_o_target(S_next, A_next, ADV_next)

                # pessimistic targets
                target_r_p = R - beta * sig + gamma * critic_r_p_target(S_next, A_next, ADV_next)
                target_u_p = C - alpha * sig + gamma * critic_u_p_target(S_next, A_next, ADV_next)

            # update critics
            loss = lambda net, tgt: ((net(S, A, ADV) - tgt)**2).mean()

            opt_r_o.zero_grad(); loss(critic_r_o, target_r_o).backward(); opt_r_o.step()
            opt_r_p.zero_grad(); loss(critic_r_p, target_r_p).backward(); opt_r_p.step()
            opt_u_o.zero_grad(); loss(critic_u_o, target_u_o).backward(); opt_u_o.step()
            opt_u_p.zero_grad(); loss(critic_u_p, target_u_p).backward(); opt_u_p.step()
            
            ##TRICK 2: Perform sequential soft-updates
            soft_update(critic_r_o, critic_r_o_target)
            soft_update(critic_r_p, critic_r_p_target)
            soft_update(critic_u_o, critic_u_o_target)
            soft_update(critic_u_p, critic_u_p_target)
            
            torch.nn.utils.clip_grad_norm_(critic_r_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_r_p.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_o.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(critic_u_p.parameters(), 1.0)
            

            # ===== Actor (optimistic) =====
            A_pred = actor(S)
            ADV_fixed = adversary(S).detach()

            q_r = critic_r_o(S, A_pred, ADV_fixed)
            q_u = critic_u_o(S, A_pred, ADV_fixed)

            actor_loss = -(q_r - lambda_ * torch.relu(q_u - b)).mean()

            opt_actor.zero_grad()
            actor_loss.backward()
            opt_actor.step()

            # ===== Adversary (pessimistic) =====
            ADV_pred = adversary(S)
            A_fixed = actor(S).detach()

            q_r = critic_r_p(S, A_fixed, ADV_pred)
            q_u = critic_u_p(S, A_fixed, ADV_pred)
            
            if t%10==0:  ## Trick 4: Use adversarial training mildly
                adv_loss = (q_r - lambda_ * torch.relu(q_u - b)).mean()
                opt_adv.zero_grad()
                adv_loss.backward()
                opt_adv.step()

            # # ===== Lambda update =====
            # with torch.no_grad():
            #     violation = critic_u_p(S, A, ADV) - b
            #     lambda_ += 0.01 * violation.mean()
            #     lambda_ = torch.clamp(lambda_, 0, 10)

        if done:
            break

    #print(f"Ep {ep}, Reward: {total_r*20:.2f}, Cost/Utility: {total_c:.2f}, Lambda: {lambda_.item():.3f}")
    reward.append(total_r*20)
    cost.append(total_c)
df = {'Reward':reward,'Cost/utility':cost}
Df = pd.DataFrame(df)
Df.to_excel('Pendulumv1_strict_violation_suggested_changes.xlsx')

100%|██████████| 1000/1000 [40:37<00:00,  2.44s/it]
